# Selection tool

In [ ]:
import numpy as np
import pyvista as pv

import mefikit as mf

pv.set_plot_theme("dark")
pv.set_jupyter_backend("static")

## Element selection expressions

Elements can be selected based on their:
- types
- ids
- dimensions

Elements can be selected based on their centroid position. The selection methods using centroids are :
- bbox
- sphere
- rectangle
- circle

As you can guess, bbox and sphere should be used for 3d meshes and rectangle and circle for 2d meshes.

Elements can be selected based on the nodes position and a boolean : whether to select element with all nodes matching the condition or any node matching the condition.
- nbbox
- nsphere
- nrect
- ncircle
- nids

Elements can be selected based on their group membership :
- group(name)
- exclude_group(name)

Elements can be selected based on their scalar fields values :
- FieldExpr > FieldExpr
- FieldExpr >= FieldExpr
- FieldExpr < FieldExpr
- FieldExpr <= FieldExpr
- FieldExpr == FieldExpr

Wherever a selector is expected, the wildcards `None`, `...` and `[:]` select every element — the explicit form is `mf.sel.all()`.


In [ ]:
sphere = mf.sel.sphere([0.5, 0.5, 0.5], 0.5)
clip_x = mf.sel.bbox([0.5, -np.inf, -np.inf], [np.inf, np.inf, np.inf])  # x > 0.5
compound_sel = ~(sphere & clip_x)

Selections are light objects, there are cheap to create and combine and independent from a support.
You should just not mix 2d selectors with 3d selectors (`sphere` vs `circle`, `bbox` vs `rectangle`).

### How does it works ?

Each objects generated by a selection function (a function from the mf.sel module) is of the `Selection` type. It implements operators so that it knows how to compose in an expression. That way an expression generates a new `SelectionExpr` which can be interpreted by the `.select(expr)` method.

In [ ]:
print(sphere)

In [ ]:
print(compound_sel)

You can see two layers of NotExpr and BinaryExpr. That is perfectly normal, it does not mean that the operation is applied twice, both NotExpr operations and both BinaryExpr op actually comes from different namespaces and it is just a form of encapsulation (first is a variant, second is an enum).

### The Selection type

Here the types manipulated are `Selection`. They are simple objects that know how to compose themselves.

In [ ]:
clip = mf.sel.bbox([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, 0.5])
sphere = mf.sel.sphere([0.5, 0.5, 0.5], 0.5)

In [ ]:
x = np.linspace(0.0, 1.0, 20, endpoint=True)
volumes = mf.build_cmesh(x, x, x)

In [ ]:
volumes.select(clip).to_mesh().to_pyvista().plot()

In [ ]:
volumes.select(sphere).to_mesh().to_pyvista().plot()

## Selection composition

One of the great strength of the select method is its composability !
Watch by yourself.

The operators `&`, `|`, `^`, `-` and `~` are available.

In [ ]:
x = np.linspace(0.0, 2.0, 100)
y = np.linspace(0.0, 1.0, 50)
faces = mf.build_cmesh(x, y)

In [ ]:
circle1 = mf.sel.circle([0.75, 0.5], 0.5)
circle2 = mf.sel.circle([1.25, 0.5], 0.5)

In [ ]:
union = faces.select(circle1 | circle2).to_mesh()

In [ ]:
pt = pv.Plotter()
pt.add_mesh(faces.descend().to_pyvista())
pt.add_mesh(union.to_pyvista())
pt.camera_position = "xy"
pt.show()

In [ ]:
intersection = faces.select(circle1 & circle2).to_mesh()

In [ ]:
pt = pv.Plotter()
pt.add_mesh(faces.descend().to_pyvista())
pt.add_mesh(intersection.to_pyvista())
pt.camera_position = "xy"
pt.show()

In [ ]:
sym_diff = faces.select(circle1 ^ circle2).to_mesh()

In [ ]:
pt = pv.Plotter()
pt.add_mesh(faces.descend().to_pyvista())
pt.add_mesh(sym_diff.to_pyvista())
pt.camera_position = "xy"
pt.show()

In [ ]:
diff = faces.select(circle1 - circle2).to_mesh()

In [ ]:
pt = pv.Plotter()
pt.add_mesh(faces.descend().to_pyvista())
pt.add_mesh(diff.to_pyvista())
pt.camera_position = "xy"
pt.show()

In [ ]:
notsel = faces.select(~circle1).to_mesh()

In [ ]:
pt = pv.Plotter()
pt.add_mesh(faces.descend().to_pyvista())
pt.add_mesh(notsel.to_pyvista())
pt.camera_position = "xy"
pt.show()

## A 3D complex example

In [ ]:
sphere = mf.sel.sphere([0.5, 0.5, 0.5], 0.5)
clip_x = mf.sel.bbox([0.5, -np.inf, -np.inf], [np.inf, np.inf, np.inf])  # x > 0.5
clip_z = mf.sel.bbox([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, 0.5])  # z < 0.5

In [ ]:
volumes.select(
    (clip_x & sphere & clip_z) | (sphere & ~clip_x & ~clip_z)
).to_mesh().to_pyvista().plot()

## Select API

`.select()` returns a lazy view: `ids()`, `len()` and reductions evaluate it on demand.

In [ ]:
volumes.select(sphere)

In [ ]:
two_quarters_expr = (clip_x & sphere & clip_z) | (sphere & ~clip_x & ~clip_z)

In [ ]:
volumes.select(two_quarters_expr).ids()

In [ ]:
len(volumes.select(two_quarters_expr))

In [ ]:
volumes.select(two_quarters_expr).mean(mf.X)

## Groups

### Groups API

Selections can be stored on the mesh as named groups. The `mesh.groups` mapping behaves like a dict: assign a selection expression (or a `{etype: ids}` dict) to create or replace a group, then manage groups with the usual operations.

In [ ]:
volumes.groups["two_quarters"] = two_quarters_expr

In [ ]:
volumes.groups

In [ ]:
volumes.groups["two_quarters"].ids()

In [ ]:
volumes.groups["two_quarters"].to_mesh()

In [ ]:
len(volumes.groups["two_quarters"])

In [ ]:
volumes.groups.rename("two_quarters", "quarter_tag")
volumes.groups

In [ ]:
del volumes.groups["quarter_tag"]
volumes.groups

### Groups inplace modifications

Groups can be modified inplace, either using `SelectionExpr` (including existing groups expr):

In [ ]:
volumes.groups["two_quarters"] = two_quarters_expr
n = len(volumes.groups["two_quarters"])
added_sel = mf.sel.bbox([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, 0.2])
volumes.groups["two_quarters"].add(added_sel)
print(len(volumes.groups["two_quarters"]), "after adding a slab (was", n, ")")

In [ ]:
volumes.groups["two_quarters"].to_mesh().to_pyvista().plot()

... or directly with element ids per element type.

In [ ]:
print(len(volumes.groups["two_quarters"]))
volumes.groups["two_quarters"].remove({"HEX8": [0, 1]})
print(len(volumes.groups["two_quarters"]))
volumes.groups["two_quarters"].add({"HEX8": [0, 1]})
print(len(volumes.groups["two_quarters"]), "back to the original size")